# 일정 관리 챗봇 만들기

이 노트북을 순서대로 따라가면 **자연어로 일정을 추가·조회·삭제할 수 있는 AI 챗봇**이 완성됩니다!

```
나: 내일 오후 3시에 팀 미팅 추가해줘
AI: 팀 미팅 일정을 추가했어요! (2026-05-14 15:00)

나: 이번 주 일정 전부 보여줘
AI: 이번 주 일정 목록입니다...
```

**완성까지의 순서**
1. 환경 설정
2. 데이터 저장 헬퍼 함수 (제공됨)
3. 에이전트 도구(Tool) 4개 — **docstring만 직접 작성**
4. 도구를 LLM 없이 직접 테스트
5. **시스템 프롬프트 직접 작성**
6. **에이전트 조립**
7. 에이전트 테스트
8. **미들웨어 직접 작성** (로깅 + 토큰 추적)

![프로젝트구조](./schedule_chatbot.png)

## Step 1. 환경 설정

In [ ]:
import json
import os
import uuid
from datetime import date

from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import wrap_tool_call, after_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime

load_dotenv("../.env")

credential_key=os.getenv("credential_key")
send_system_name=os.getenv("send_system_name")
model=os.getenv("model")
api_base_url=os.getenv("api_base_url")
user_id=os.getenv("user_id")

os.environ["OPENAI_API_KEY"] = 'api_key'

model = ChatOpenAI(
    model=model,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticekt': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': "AD_ID",
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0.7,
)

## Step 2. 데이터 저장 헬퍼 함수 (제공됨)

일정을 `data/schedules.json` 파일에 저장하고 불러오는 내부 함수입니다.

`@tool`로 노출하지 않고 아래 도구들 내부에서만 사용합니다.

In [ ]:
DATA_PATH = "./data/schedules.json"

def _load_schedules():
    if not os.path.exists(DATA_PATH):
        return []
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def _save_schedules(schedules):
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    with open(DATA_PATH, "w", encoding="utf-8") as f:
        json.dump(schedules, f, ensure_ascii=False, indent=2)

## Step 3. 도구(Tool) 만들기

구현부는 제공되어 있습니다. **docstring만 직접 작성하세요.**

docstring은 단순한 설명이 아닙니다 — LLM이 이 텍스트를 읽고 *언제, 어떻게* 이 도구를 사용할지 결정합니다.

잘 쓴 docstring과 대충 쓴 docstring이 에이전트 동작에 어떤 차이를 만드는지 Step 4에서 직접 확인해 보세요.

In [ ]:
@tool
def add_schedule(date: str, time: str, title: str, description: str = "") -> str:
    """TODO: 이 도구가 무엇을 하는지, 어떤 인자를 받는지 설명하세요."""
    schedules = _load_schedules()
    new_schedule = {
        "id": str(uuid.uuid4())[:8],
        "date": date,
        "time": time,
        "title": title,
        "description": description,
    }
    schedules.append(new_schedule)
    _save_schedules(schedules)
    return f"일정이 추가되었습니다! (ID: {new_schedule['id']}) {date} {time} - {title}"

In [ ]:
@tool
def get_schedules(date: str) -> str:
    """TODO: 이 도구가 무엇을 하는지, 어떤 인자를 받는지 설명하세요."""
    schedules = _load_schedules()
    day_schedules = [s for s in schedules if s["date"] == date]
    if not day_schedules:
        return f"{date}에 등록된 일정이 없습니다."
    day_schedules.sort(key=lambda x: x["time"])
    result = f"[{date} 일정 목록]\n"
    for s in day_schedules:
        result += f"- {s['time']} {s['title']}"
        if s["description"]:
            result += f" ({s['description']})"
        result += f" [ID: {s['id']}]\n"
    return result

In [ ]:
@tool
def list_all_schedules() -> str:
    """TODO: 이 도구가 무엇을 하는지 설명하세요."""
    schedules = _load_schedules()
    if not schedules:
        return "저장된 일정이 없습니다."
    schedules.sort(key=lambda x: (x["date"], x["time"]))
    result = "[전체 일정]\n"
    current_date = ""
    for s in schedules:
        if s["date"] != current_date:
            current_date = s["date"]
            result += f"\n📅 {current_date}\n"
        result += f"  - {s['time']} {s['title']}"
        if s["description"]:
            result += f" ({s['description']})"
        result += f" [ID: {s['id']}]\n"
    return result

In [ ]:
@tool
def delete_schedule(schedule_id: str) -> str:
    """TODO: 이 도구가 무엇을 하는지, 어떤 인자를 받는지 설명하세요."""
    schedules = _load_schedules()
    target = next((s for s in schedules if s["id"] == schedule_id), None)
    if not target:
        return f"ID '{schedule_id}'에 해당하는 일정을 찾을 수 없습니다. 먼저 일정을 조회해서 ID를 확인해주세요."
    schedules.remove(target)
    _save_schedules(schedules)
    return f"'{target['title']}' 일정이 삭제되었습니다. ({target['date']} {target['time']})"

## Step 4. 도구 직접 테스트 (LLM 없이)

에이전트를 만들기 전에 도구들이 제대로 동작하는지 먼저 확인합니다.

`@tool` 함수는 `.invoke()`로 직접 호출할 수 있습니다.

In [ ]:
print(add_schedule.invoke({"date": "2026-05-20", "time": "10:00", "title": "팀 미팅"}))
print(add_schedule.invoke({"date": "2026-05-20", "time": "14:00", "title": "점심 약속", "description": "삼겹살 식당"}))
print(add_schedule.invoke({"date": "2026-05-21", "time": "09:00", "title": "조기 축구"}))

In [ ]:
print(get_schedules.invoke({"date": "2026-05-20"}))

In [ ]:
print(list_all_schedules.invoke({}))

## Step 5. 시스템 프롬프트 작성

에이전트의 행동 지침을 직접 작성해 보세요.

아래 4가지를 포함하면 좋습니다:
1. **역할 정의** - 이 AI가 무엇을 하는 어시스턴트인지
2. **도구 사용 규칙** - 어떤 상황에 어떤 도구를 써야 하는지
3. **데이터 형식** - 날짜/시간 형식 규칙
4. **답변 규칙** - 어떻게 답변해야 하는지

> 에이전트가 '오늘', '내일', '이번 주' 같은 표현을 처리하려면 오늘 날짜를 프롬프트에 넣어야 합니다.

In [ ]:
today_str = date.today().strftime("%Y-%m-%d")

system_prompt = f"""
# TODO: 시스템 프롬프트를 작성하세요
"""

## Step 6. 에이전트 만들기

`create_agent()`로 도구, 시스템 프롬프트, 메모리를 묶어 에이전트를 만드세요.

In [ ]:
agent = create_agent(
    # TODO
)

## Step 7. 에이전트 테스트

다양한 명령을 입력해서 에이전트가 의도한 대로 동작하는지 확인하세요.

만약 엉뚱한 도구를 사용하거나 날짜 계산이 틀린다면 → Step 5의 시스템 프롬프트 또는 Step 3의 docstring을 수정해 보세요.

In [ ]:
response = agent.invoke(
    {"messages": {"role": "user", "content": "내일 오후 3시에 팀 미팅 추가해줘"}},
    {"configurable": {"thread_id": "1"}},
)
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": {"role": "user", "content": "내일 일정이 뭐야?"}},
    {"configurable": {"thread_id": "1"}},
)
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": {"role": "user", "content": "방금 추가한 팀 미팅 삭제해줘"}},
    {"configurable": {"thread_id": "1"}},
)
print(response["messages"][-1].content)

## Step 8. 미들웨어 추가

미들웨어란 에이전트가 작업을 처리하기 전/후에 개입하는 기능입니다.

두 가지를 직접 작성하세요:
1. 도구가 호출될 때마다 이름과 인수를 출력하는 미들웨어

In [ ]:
# ???
def tool_logger(request, handler):
    # request.tool_call["name"] : 도구 이름
    # request.tool_call["args"] : 도구 인수
    # handler(request)          : 실제 도구 실행 → ToolMessage 반환
    # result.content            : 도구 실행 결과 문자열
    pass

2. 응답이 완료될 때마다 토큰 사용량을 출력

In [ ]:
_total = {"input": 0, "output": 0}

# ???
def token_tracker(state: AgentState, runtime: Runtime):
    last = state["messages"][-1]
    if not (hasattr(last, "usage_metadata") and last.usage_metadata):
        return None

    u = last.usage_metadata
    inp = u.get("input_tokens", 0)
    out = u.get("output_tokens", 0)
    _total["input"] += inp
    _total["output"] += out
    print(f"  [TOKEN] 누적: 입력 {_total['input']} / 출력 {_total['output']} / 합계 {_total['input'] + _total['output']}")
    return None

In [ ]:
agent = create_agent(
    # TODO: 위에서 만든 미들웨어를 포함해서 에이전트를 다시 만드세요
)

In [ ]:
test_messages = [
    "모레 오전 10시에 치과 예약 추가해줘",
    "이번 주 전체 일정 보여줘",
]

for msg in test_messages:
    print(f"나: {msg}")
    response = agent.invoke(
        {"messages": {"role": "user", "content": msg}},
        {"configurable": {"thread_id": "test"}},
    )
    print(f"AI: {response['messages'][-1].content}\n")

print("=" * 50)
print(f"[최종 토큰 합계] 입력 {_total['input']} / 출력 {_total['output']} / 합계 {_total['input'] + _total['output']}")